# Imports

In [3]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import faiss
import time

# Config

In [ ]:
BASE_DIR = os.path.join(os.getcwd(), "TFE_Data")
CURRENT_DATASET = "Flickr8k"

MULTIMODAL = ["openclip_l14", "clip"]

FUSION_EXPORT = os.path.join(BASE_DIR, "Fusion_Multimodal")
os.makedirs(FUSION_EXPORT, exist_ok=True)


# Load Data

In [5]:
def load_raw(modality, model):
    path = os.path.join(
        BASE_DIR, "Results_Multimodal", CURRENT_DATASET,
        model, f"{modality}_embeddings.npy"
    )
    return np.load(path)


In [6]:
def load_reduced(modality, model, method, dim):
    path = os.path.join(
        BASE_DIR, "Results_Multimodal", CURRENT_DATASET,
        model, "Features_Reduced", modality, method,
        f"X_{modality}_{model}_{method}_{dim}_{CURRENT_DATASET}.npy"
    )
    return np.load(path)


In [7]:
def repeat_vision_embeddings(V, T):
    """
    Repeat each vision embedding to match the number of text embeddings.
    Assumes 5 captions per image (Flickr8k/Flickr30k).
    """
    captions_per_image = len(T) // len(V)
    return np.repeat(V, captions_per_image, axis=0)


# FAISS

In [8]:
def normalize_faiss(X):
    X = X.astype("float32")
    faiss.normalize_L2(X)
    return X

def faiss_top50(V, T):
    """Fast retrieval: top-50 only."""
    V = normalize_faiss(V)
    T = normalize_faiss(T)
    index = faiss.IndexFlatIP(T.shape[1])
    index.add(T)
    sims, idx = index.search(V, 50)
    return idx

def faiss_full_ranking(V, T):
    """Full ranking: used only for best variants."""
    V = normalize_faiss(V)
    T = normalize_faiss(T)
    index = faiss.IndexFlatIP(T.shape[1])
    index.add(T)
    sims, idx = index.search(V, T.shape[0])
    return idx

In [9]:
def retrieval_metrics_top50(indices):
    N = len(indices)
    return {
        "R@1":  np.mean([i in indices[i][:1] for i in range(N)]),
        "R@5":  np.mean([i in indices[i][:5] for i in range(N)]),
        "R@10": np.mean([i in indices[i][:10] for i in range(N)]),
        "R@50": np.mean([i in indices[i][:50] for i in range(N)]),
    }

def retrieval_metrics_full(indices):
    N = len(indices)
    ranks = []
    for i in range(N):
        rank = np.where(indices[i] == i)[0][0] + 1
        ranks.append(rank)
    ranks = np.array(ranks)
    return {
        "MnR": np.mean(ranks),
        "MedR": np.median(ranks)
    }

# Reduction Methods

In [10]:
def fuse_concat(V, T): return np.concatenate([V, T], axis=1)
def fuse_add(V, T): return V + T
def fuse_gated(V, T, a=0.5): return a*V + (1-a)*T

def fuse_simweighted(V, T):
    sim = (V * T).sum(axis=1, keepdims=True)
    w = sim / (sim + 1e-8)
    return w * V + (1 - w) * T

In [11]:
results = []

for model in MULTIMODAL:
    print(f"\n=== {model.upper()} ===")

    V_raw = load_raw("vision", model)
    T_raw = load_raw("text", model)
    V_raw = repeat_vision_embeddings(V_raw, T_raw)

    # -------------------------
    # RAW FUSION
    # -------------------------
    print("Raw fusion: concat")
    concat_raw = fuse_concat(V_raw, T_raw)
    concat_raw_red = PCA(n_components=T_raw.shape[1]).fit_transform(concat_raw)
    idx = faiss_top50(concat_raw_red, T_raw)
    results.append({"model": model, "fusion": "raw_concat", **retrieval_metrics_top50(idx)})
    np.save(os.path.join(FUSION_EXPORT, f"{model}_raw_concat.npy"), concat_raw_red)

    print("Raw fusion: add")
    fused_add = fuse_add(V_raw, T_raw)
    idx = faiss_top50(fused_add, T_raw)
    results.append({"model": model, "fusion": "raw_add", **retrieval_metrics_top50(idx)})
    np.save(os.path.join(FUSION_EXPORT, f"{model}_raw_add.npy"), fused_add)

    print("Raw fusion: gated")
    fused_gated = fuse_gated(V_raw, T_raw)
    idx = faiss_top50(fused_gated, T_raw)
    results.append({"model": model, "fusion": "raw_gated", **retrieval_metrics_top50(idx)})
    np.save(os.path.join(FUSION_EXPORT, f"{model}_raw_gated.npy"), fused_gated)

    print("Raw fusion: simweighted")
    fused_sim = fuse_simweighted(V_raw, T_raw)
    idx = faiss_top50(fused_sim, T_raw)
    results.append({"model": model, "fusion": "raw_simweighted", **retrieval_metrics_top50(idx)})
    np.save(os.path.join(FUSION_EXPORT, f"{model}_raw_simweighted.npy"), fused_sim)

    # -------------------------
    # REDUCE → FUSE
    # -------------------------
    reduced_root = os.path.join(
        BASE_DIR, "Results_Multimodal", CURRENT_DATASET,
        model, "Features_Reduced"
    )
    if not os.path.exists(reduced_root):
        continue

    for method in os.listdir(os.path.join(reduced_root, "vision")):
        for dim in DIMENSIONS:
            try:
                V_red = load_reduced("vision", model, method, dim)
                T_red = load_reduced("text", model, method, dim)
                V_red = repeat_vision_embeddings(V_red, T_red)
            except:
                continue

            print(f"{model} | {method} | dim={dim}")

            # CONCAT
            concat_red = fuse_concat(V_red, T_red)
            concat_red_red = PCA(n_components=dim).fit_transform(concat_red)
            idx = faiss_top50(concat_red_red, T_red)
            results.append({
                "model": model,
                "fusion": "reduce_then_fuse_concat",
                "method": method,
                "dim": dim,
                **retrieval_metrics_top50(idx)
            })
            np.save(os.path.join(FUSION_EXPORT, f"{model}_{method}_{dim}_concat.npy"), concat_red_red)

            # ADD
            fused_add = fuse_add(V_red, T_red)
            idx = faiss_top50(fused_add, T_red)
            results.append({
                "model": model,
                "fusion": "reduce_then_fuse_add",
                "method": method,
                "dim": dim,
                **retrieval_metrics_top50(idx)
            })
            np.save(os.path.join(FUSION_EXPORT, f"{model}_{method}_{dim}_add.npy"), fused_add)

            # GATED
            fused_gated = fuse_gated(V_red, T_red)
            idx = faiss_top50(fused_gated, T_red)
            results.append({
                "model": model,
                "fusion": "reduce_then_fuse_gated",
                "method": method,
                "dim": dim,
                **retrieval_metrics_top50(idx)
            })
            np.save(os.path.join(FUSION_EXPORT, f"{model}_{method}_{dim}_gated.npy"), fused_gated)

            # SIMWEIGHTED
            fused_sim = fuse_simweighted(V_red, T_red)
            idx = faiss_top50(fused_sim, T_red)
            results.append({
                "model": model,
                "fusion": "reduce_then_fuse_simweighted",
                "method": method,
                "dim": dim,
                **retrieval_metrics_top50(idx)
            })
            np.save(os.path.join(FUSION_EXPORT, f"{model}_{method}_{dim}_simweighted.npy"), fused_sim)



=== OPENCLIP_L14 ===
Raw fusion: concat
Raw fusion: add
Raw fusion: gated
Raw fusion: simweighted
openclip_l14 | svd | dim=512
openclip_l14 | svd | dim=256
openclip_l14 | svd | dim=128
openclip_l14 | svd | dim=64
openclip_l14 | svd | dim=32
openclip_l14 | svd | dim=16
openclip_l14 | umap | dim=512
openclip_l14 | umap | dim=256
openclip_l14 | umap | dim=128
openclip_l14 | umap | dim=64
openclip_l14 | umap | dim=32
openclip_l14 | umap | dim=16
openclip_l14 | pca | dim=512
openclip_l14 | pca | dim=256
openclip_l14 | pca | dim=128
openclip_l14 | pca | dim=64
openclip_l14 | pca | dim=32
openclip_l14 | pca | dim=16
openclip_l14 | grp | dim=512
openclip_l14 | grp | dim=256
openclip_l14 | grp | dim=128
openclip_l14 | grp | dim=64
openclip_l14 | grp | dim=32
openclip_l14 | grp | dim=16

=== EVA ===
Raw fusion: concat
Raw fusion: add
Raw fusion: gated
Raw fusion: simweighted
eva | svd | dim=512
eva | svd | dim=256
eva | svd | dim=128
eva | svd | dim=64
eva | svd | dim=32
eva | svd | dim=16
eva 

In [12]:
df_fusion = pd.DataFrame(results)
df_fusion.to_csv(os.path.join(BASE_DIR, "Fusion_Multimodal_Results.csv"), index=False)
print("Saved fast metrics.")


Saved fast metrics.


<!-- ## Silhouette Score -->